# Phase D 

### la dataset 

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

def charger_sonar(url):
    

    donnees = pd.read_csv(
        url,
        header=None
    )

    X = donnees.iloc[:, :-1].copy()

    X.columns = [
        f"frequence_{numero}"
        for numero in range(1, 61)
    ]

    y = donnees.iloc[:, -1].map({
        "M": 1,
        "R": 0
    })

    nombre_mines = (y == 1).sum()
    nombre_rochers = (y == 0).sum()

    print(
        f"Sonar : {X.shape}, "
        f"mines={nombre_mines}, "
        f"rochers={nombre_rochers}"
    )

    return X, y
X, y = charger_sonar("sonar.all-data")

print("\nValeurs manquantes dans X :", X.isna().sum().sum())
print("Valeurs manquantes dans y :", y.isna().sum())


### séparation entraînement/test

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nRépartition des classes dans le jeu de test :")
print(y_test.value_counts().sort_index())


Répartition des classes dans le jeu de test :
60
0    20
1    22
Name: count, dtype: int64


#### Cas normal

#### la régression logistique

In [9]:
modele_logistique = Pipeline([
    (
        "standardisation",
        StandardScaler()
    ),
    (
        "modele",
        LogisticRegression(
            max_iter=5000,
            random_state=42
        )
    )
])
modele_logistique.fit(
    X_train,
    y_train
)

predictions_logistique = modele_logistique.predict(
    X_test
)

accuracy_logistique = accuracy_score(
    y_test,
    predictions_logistique
)

print(
    "LogisticRegression : "
    f"accuracy={accuracy_logistique:.2f}"
)

LogisticRegression : accuracy=0.83


#### SVM avec noyau RBF

In [10]:
modele_svm = Pipeline([
    (
        "standardisation",
        StandardScaler()
    ),
    (
        "modele",
        SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        )
    )
])
modele_svm.fit(
    X_train,
    y_train
)

predictions_svm = modele_svm.predict(
    X_test
)

accuracy_svm = accuracy_score(
    y_test,
    predictions_svm
)

print(
    "SVC (rbf) : "
    f"accuracy={accuracy_svm:.2f}"
)

SVC (rbf) : accuracy=0.93


#### Random Forest

In [11]:
modele_random_forest = Pipeline([
    (
        "standardisation",
        StandardScaler()
    ),
    (
        "modele",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    )
])
modele_random_forest.fit(
    X_train,
    y_train
)

predictions_random_forest = modele_random_forest.predict(
    X_test
)

accuracy_random_forest = accuracy_score(
    y_test,
    predictions_random_forest
)

print(
    "RandomForest : "
    f"accuracy={accuracy_random_forest:.2f}"
)

RandomForest : accuracy=0.83


### comparaison des resultats 

In [12]:
resultats_normaux = pd.DataFrame({
    "Modèle": [
        "LogisticRegression",
        "SVC (rbf)",
        "RandomForest"
    ],
    "Accuracy": [
        accuracy_logistique,
        accuracy_svm,
        accuracy_random_forest
    ]
})

resultats_normaux = resultats_normaux.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print("Comparaison des modèles — Cas normal :")

display(resultats_normaux.round(3))

Comparaison des modèles — Cas normal :


,Modèle,Accuracy
0,SVC (rbf),0.929
1,LogisticRegression,0.833
2,RandomForest,0.833


#### Cas limite

#### la régression logistique sans standardisation

In [14]:
logistique_sans_standardisation = LogisticRegression(
    max_iter=5000,
    random_state=42
)
logistique_sans_standardisation.fit(
    X_train,
    y_train
)

predictions_logistique_sans = (
    logistique_sans_standardisation.predict(X_test)
)

accuracy_logistique_sans = accuracy_score(
    y_test,
    predictions_logistique_sans
)

print(
    "LogisticRegression sans standardisation : "
    f"accuracy={accuracy_logistique_sans:.2f}"
)

LogisticRegression sans standardisation : accuracy=0.81


#### SVM sans standardisation

In [15]:
svm_sans_standardisation = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)
svm_sans_standardisation.fit(
    X_train,
    y_train
)

predictions_svm_sans = svm_sans_standardisation.predict(
    X_test
)

accuracy_svm_sans = accuracy_score(
    y_test,
    predictions_svm_sans
)

print(
    "SVC sans standardisation : "
    f"accuracy={accuracy_svm_sans:.2f}"
)

SVC sans standardisation : accuracy=0.83


#### Random Forest sans standardisation

In [16]:
random_forest_sans_standardisation = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
random_forest_sans_standardisation.fit(
    X_train,
    y_train
)

predictions_forest_sans = (
    random_forest_sans_standardisation.predict(X_test)
)

accuracy_forest_sans = accuracy_score(
    y_test,
    predictions_forest_sans
)

print(
    "RandomForest sans standardisation : "
    f"accuracy={accuracy_forest_sans:.2f}"
)

RandomForest sans standardisation : accuracy=0.83


#### comparaison avec et sans standardisation

In [17]:
comparaison_standardisation = pd.DataFrame({
    "Modèle": [
        "LogisticRegression",
        "SVC (rbf)",
        "RandomForest"
    ],
    "Avec standardisation": [
        accuracy_logistique,
        accuracy_svm,
        accuracy_random_forest
    ],
    "Sans standardisation": [
        accuracy_logistique_sans,
        accuracy_svm_sans,
        accuracy_forest_sans
    ]
})

comparaison_standardisation["Écart"] = (
    comparaison_standardisation["Avec standardisation"]
    - comparaison_standardisation["Sans standardisation"]
)

print("Comparaison avec et sans standardisation :")

display(comparaison_standardisation.round(3))

Comparaison avec et sans standardisation :


,Modèle,Avec standardisation,Sans standardisation,Écart
0,LogisticRegression,0.833,0.810,0.024
1,SVC (rbf),0.929,0.833,0.095
2,RandomForest,0.833,0.833,0.000


#### Cas adversarial

#### création du signal nul

In [20]:
echo_zero = pd.DataFrame(
    np.zeros((1, 60)),
    columns=X.columns
)

display(echo_zero)

,frequence_1,frequence_2,frequence_3,frequence_4,frequence_5,frequence_6,frequence_7,frequence_8,frequence_9,frequence_10,...,frequence_51,frequence_52,frequence_53,frequence_54,frequence_55,frequence_56,frequence_57,frequence_58,frequence_59,frequence_60
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### prédiction de la régression logistique 

In [21]:
prediction_zero_logistique = modele_logistique.predict(
    echo_zero
)[0]

probabilites_zero_logistique = (
    modele_logistique.predict_proba(echo_zero)[0]
)

confiance_zero_logistique = (
    probabilites_zero_logistique.max()
)

classe_zero_logistique = (
    "mine"
    if prediction_zero_logistique == 1
    else "rocher"
)

print("Régression logistique :")
print("Classe prédite :", classe_zero_logistique)

print(
    "Confiance :",
    round(confiance_zero_logistique, 3)
)

Régression logistique :
Classe prédite : rocher
Confiance : 1.0


#### prédiction du SVM 

In [22]:
prediction_zero_svm = modele_svm.predict(
    echo_zero
)[0]

probabilites_zero_svm = modele_svm.predict_proba(
    echo_zero
)[0]

confiance_zero_svm = probabilites_zero_svm.max()

classe_zero_svm = (
    "mine"
    if prediction_zero_svm == 1
    else "rocher"
)

print("SVM avec noyau RBF :")
print("Classe prédite :", classe_zero_svm)

print(
    "Confiance :",
    round(confiance_zero_svm, 3)
)

SVM avec noyau RBF :
Classe prédite : rocher
Confiance : 0.661


#### prédiction du Random Forest

In [23]:
prediction_zero_forest = modele_random_forest.predict(
    echo_zero
)[0]

probabilites_zero_forest = (
    modele_random_forest.predict_proba(echo_zero)[0]
)

confiance_zero_forest = (
    probabilites_zero_forest.max()
)

classe_zero_forest = (
    "mine"
    if prediction_zero_forest == 1
    else "rocher"
)

print("Random Forest :")
print("Classe prédite :", classe_zero_forest)

print(
    "Confiance :",
    round(confiance_zero_forest, 3)
)

Random Forest :
Classe prédite : rocher
Confiance : 0.715


#### contrôle avant la prédiction

In [24]:
if np.allclose(echo_zero.to_numpy(), 0):

    print(
        "Signal invalide : toutes les mesures "
        "sont égales à zéro."
    )

    print(
        "La prédiction doit être refusée et "
        "le capteur doit être vérifié."
    )

else:

    print(
        "Le signal peut être envoyé au modèle."
    )

Signal invalide : toutes les mesures sont égales à zéro.
La prédiction doit être refusée et le capteur doit être vérifié.
